In [ ]:
# Eugenio RUggieri
# 2026-05-05
# This script is used to verify the Chroma_DB Collection locally stored in the /mnt/f/chroma_db_store

In [1]:
import os
import chromadb
from chromadb.config import Settings
from pathlib import Path

# Path esatto mappato sul tuo SSD esterno Crucial X9
CHROMA_DB_PATH = "/mnt/f/chroma_db_store"

def analyze_chroma_architecture(db_path: str):
    path_obj = Path(db_path)
    if not path_obj.exists():
        print(f"❌ Errore: Il path {db_path} non esiste.")
        return

    print(f"🔍 Avvio Analisi Architetturale Ibrida su: {db_path}\n")
    print("=" * 60)
    
    # --- 1. ANALISI FISICA DELLO STORAGE (I/O SSD) ---
    print("💾 [1] ANALISI FISICA STORAGE (SSD Crucial X9)")
    total_size_mb = 0
    
    for item in path_obj.iterdir():
        if item.is_file():
            size_mb = item.stat().st_size / (1024 * 1024)
            total_size_mb += size_mb
            if item.name == "chroma.sqlite3":
                print(f"   ▪️ Database Testi/Metadati ({item.name}): {size_mb:.2f} MB")
        elif item.is_dir():
            dir_size_mb = sum(f.stat().st_size for f in item.rglob('*') if f.is_file()) / (1024 * 1024)
            total_size_mb += dir_size_mb
            print(f"   ▪️ Indice Vettoriale (HNSW) [{item.name[:8]}...]: {dir_size_mb:.2f} MB")
            
    print(f"   => Spazio Totale Occupato su Disco: {total_size_mb / 1024:.2f} GB\n")
    print("-" * 60)

    # --- 2. ANALISI LOGICA (API ChromaDB) ---
    print("🧠 [2] ANALISI LOGICA COLLEZIONI (ChromaDB API)")
    try:
        # FIX 1: Disabilitazione Telemetria per rimuovere i warning di PostHog
        client = chromadb.PersistentClient(
            path=db_path,
            settings=Settings(anonymized_telemetry=False)
        )
        collections = client.list_collections()
        
        if not collections:
            print("   ⚠️ Nessuna collezione logica trovata.")
            return

        for coll_ref in collections:
            coll_name = coll_ref.name if hasattr(coll_ref, 'name') else coll_ref
            collection = client.get_collection(name=coll_name)
            
            record_count = collection.count()
            metadata = collection.metadata or "Default"
            
            emb_dim = "N/A"
            if record_count > 0:
                sample = collection.peek(1)
                embeddings = sample.get('embeddings')
                
                # FIX 2: Controllo sicuro per Array Numpy, evita l'errore di "ambiguous truth value"
                if embeddings is not None and len(embeddings) > 0:
                    emb_dim = len(embeddings[0])
            
            print(f"   📂 Collezione: \033[1m{coll_name}\033[0m")
            print(f"      ▪️ Record (Chunk): {record_count:,}")
            print(f"      ▪️ Dimensionalità Vettori: {emb_dim}")
            print(f"      ▪️ Configurazione Spaziale: {metadata}")
            
    except Exception as e:
        print(f"❌ Errore durante la connessione logica a ChromaDB: {e}")
        
    print("=" * 60)

if __name__ == "__main__":
    analyze_chroma_architecture(CHROMA_DB_PATH)

🔍 Avvio Analisi Architetturale Ibrida su: /mnt/f/chroma_db_store

💾 [1] ANALISI FISICA STORAGE (SSD Crucial X9)
   ▪️ Indice Vettoriale (HNSW) [7bf1a857...]: 0.00 MB
   ▪️ Database Testi/Metadati (chroma.sqlite3): 8632.64 MB
   ▪️ Indice Vettoriale (HNSW) [082178b8...]: 8.37 MB
   ▪️ Indice Vettoriale (HNSW) [7a69bef0...]: 8.37 MB
   ▪️ Indice Vettoriale (HNSW) [c244748b...]: 25.30 MB
   ▪️ Indice Vettoriale (HNSW) [887e3cb7...]: 0.16 MB
   ▪️ Indice Vettoriale (HNSW) [54434551...]: 44.55 MB
   ▪️ Indice Vettoriale (HNSW) [d5d31050...]: 291.81 MB
   ▪️ Indice Vettoriale (HNSW) [2bc1dd20...]: 1290.79 MB
   => Spazio Totale Occupato su Disco: 10.06 GB

------------------------------------------------------------
🧠 [2] ANALISI LOGICA COLLEZIONI (ChromaDB API)


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


   📂 Collezione: internal_knowledge_backup_old
      ▪️ Record (Chunk): 25,681
      ▪️ Dimensionalità Vettori: 384
      ▪️ Configurazione Spaziale: Default
   📂 Collezione: vision_cases
      ▪️ Record (Chunk): 194
      ▪️ Dimensionalità Vettori: 384
      ▪️ Configurazione Spaziale: Default


Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


   📂 Collezione: drugs_knowledge
      ▪️ Record (Chunk): 16,029
      ▪️ Dimensionalità Vettori: 384
      ▪️ Configurazione Spaziale: Default
   📂 Collezione: internal_knowledge
      ▪️ Record (Chunk): 782,154
      ▪️ Dimensionalità Vettori: 384
      ▪️ Configurazione Spaziale: Default


## ATTENZIONE - Codice seguente da NON RILANCIARE. Usato ESCLUSIVAMENTE per scambiare IMB-MCQA con IMB-QA nel DB_CHROME Locale
## Adesso IMB-MCQA == Collezione: internal_knowledge_backup_old
## Adesso IMB-QA == Collezione: internal_knowledge
##        DART ==  Collezione: drugs_knowledge
##        MMMED ==  Collezione: Collezione: vision_cases

In [1]:
import chromadb

# --- CONFIGURAZIONE ---
CHROMA_DB_PATH = "/mnt/f/chroma_db_store"
client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

print("⚙️ Inizio procedura di swap sicuro delle collezioni...")

try:
    # STEP 1: Mettiamo al sicuro la vecchia collezione piccola (se esiste)
    try:
        vecchia_collezione = client.get_collection(name="internal_knowledge")
        vecchia_collezione.modify(name="internal_knowledge_backup_old")
        conteggio_vecchia = vecchia_collezione.count()
        print(f"✅ STEP 1: Pre-esistente 'internal_knowledge' ({conteggio_vecchia} record) rinominata in 'internal_knowledge_backup_old'.")
    except ValueError:
        print("ℹ️ STEP 1 Skip: Nessuna collezione 'internal_knowledge' preesistente trovata nel DB.")

    # STEP 2: Promuoviamo la nuova collezione massiva
    nuova_collezione_massiva = client.get_collection(name="external_knowledge")
    conteggio_nuova = nuova_collezione_massiva.count()
    nuova_collezione_massiva.modify(name="internal_knowledge")
    print(f"✅ STEP 2: 'external_knowledge' ({conteggio_nuova} record) promossa a 'internal_knowledge'.")
    
    print("-" * 60)
    print("🚀 OPERAZIONE COMPLETATA CON SUCCESSO!")
    print("Ora il tuo codice Python della Cella 3 punterà automaticamente alla collezione da 780k+ record senza accorgersi dello scambio.")
    
except Exception as e:
    print(f"❌ ERRORE: {str(e)}")
    print("Stato attuale delle collezioni:", [c.name for c in client.list_collections()])

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


⚙️ Inizio procedura di swap sicuro delle collezioni...
✅ STEP 1: Pre-esistente 'internal_knowledge' (25681 record) rinominata in 'internal_knowledge_backup_old'.
✅ STEP 2: 'external_knowledge' (782154 record) promossa a 'internal_knowledge'.
------------------------------------------------------------
🚀 OPERAZIONE COMPLETATA CON SUCCESSO!
Ora il tuo codice Python della Cella 3 punterà automaticamente alla collezione da 780k+ record senza accorgersi dello scambio.
